# AB-SAW 抵抗解析の初期観測

既存の生成モデルと抵抗ソルバーを使い、占有率・系サイズごとに試料を生成して5種類の図を作ります。コードはGitHub、入力と結果はDriveに保存します。

最初は `MODE = "smoke"` で12試料を確認します。その後、`MODE = "full"` に変更し、上から再実行すると、L=32,64,128 × 占有率0.10〜0.70（0.01刻み）× 各100試料を測定します。全18,300試料です。

## 1. Driveに接続し、実行条件を選択

`RUN_ID` をそろえると、実行済み試料を再利用して再開します。条件を変える新しい実験では別名を指定してください。サイズを段階的に測る場合は、例えば `SIZE_OVERRIDE = [32]` とし、`RUN_ID` も変更してください。

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
from tempfile import mkdtemp
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import drive
from IPython.display import Image, display

drive.mount("/content/drive")

MODE = "smoke"  # smoke / full
RUN_ID = "initial100_v1"
SIZE_OVERRIDE = None  # 例 [32]、全サイズなら None
RETRY_FAILED = False

if MODE not in {"smoke", "full"}:
    raise ValueError("MODE must be smoke or full")
if Path(RUN_ID).name != RUN_ID or RUN_ID in {"", ".", ".."}:
    raise ValueError("RUN_IDにはフォルダ名だけを指定してください")
drive_root = Path("/content/drive/MyDrive/tetra-peg-network/resistance_analysis")
output = drive_root / "studies" / RUN_ID / MODE
print("出力先", output)


## 2. GitHubからコードを取得し、Cの生成プログラムをビルド

In [ ]:
CODE_REF = "0cf226696fdce8f275e4ac9cb98fcc7c3ffb6595"
REPOSITORY = "https://github.com/takakei510/tetra-peg-network.git"
work = Path(mkdtemp(prefix="resistance-study-", dir="/content"))
repo = work / "tetra-peg-network"
subprocess.run(["git", "init", "-q", str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", REPOSITORY, CODE_REF], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "-q", "--detach", "FETCH_HEAD"], check=True)
subprocess.run(["make", "build/demo_network"], cwd=repo, check=True)
try:
    import numpy
    import scipy
    import matplotlib
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy", "scipy", "matplotlib"], check=True)
print("コードのコミット", CODE_REF)


## 3. 設定を確認して、試料の生成と抵抗解析を実行

全試料で同じ電極規則を使います。正常試料の貫通率と平均Gを集計し、非連結や電極接点なしは集団平均のGに0を与えます。元データの状態は保持します。生成未達や計算失敗は除外件数を記録します。

分子粗視化抵抗網を再計算できるよう、分子コア座標と成立結合、各枝電流・電位を圧縮CSVで保存します。全腕のSAW経路は保存対象に含みません。

In [ ]:
config_name = "resistance-study-smoke.json" if MODE == "smoke" else "resistance-study-100.json"
config = json.loads((repo / "configs/analysis" / config_name).read_text(encoding="utf-8"))
if SIZE_OVERRIDE is not None:
    config["sizes"] = SIZE_OVERRIDE
config_path = work / "study-config.json"
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
count = len(config["sizes"]) * len(config["phi_percent"]) * len(config["seeds"])
print("サイズ", config["sizes"], "条件ごとの試料数", len(config["seeds"]), "合計", count, flush=True)
print("出力先", output, flush=True)

command = [sys.executable, str(repo / "scripts/analysis/resistance_study.py"),
           "--config", str(config_path), "--output", str(output),
           "--generator", str(repo / "build/demo_network")]
if RETRY_FAILED:
    command.append("--retry-failed")
with subprocess.Popen(command, cwd=repo, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, bufsize=1) as process:
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        code = process.wait()
        if code != 0:
            raise subprocess.CalledProcessError(code, command)
    finally:
        if process.poll() is None:
            process.terminate()
            process.wait()


## 4. 図を作成して確認

全図は点と誤差棒で表示します。貫通率は95% Wilson区間、平均Gは±1標準誤差、抵抗は連結試料のみの中央値と四分位範囲です。抵抗図の下段で連結試料数を確認します。各条件100試料に達したかは集計CSVの `n_valid` で確認してください。

`smoke` の図は各2試料の動作確認結果です。

In [ ]:
subprocess.run([sys.executable, str(repo / "scripts/analysis/plot_resistance_study.py"), str(output)],
               cwd=repo, check=True)
for name in ("01_spanning", "02_conductance", "03_resistance", "04_current_carrying", "05_power_participation"):
    display(Image(filename=str(output / "figures" / (name + ".png"))))
print((output / "resistance_summary.csv").read_text(encoding="utf-8-sig")[:2500])


## 5. 結果を共有するZIPを作成

このZIPには集計CSV、設定、図を入れます。試料ごとの入力・電流・電位はDriveの `cases` に残ります。ZIPをこのチャットへ添付すると、観測結果と統計・数値精度を確認できます。

In [ ]:
archive = output / "resistance_report.zip"
with ZipFile(archive, "w", compression=ZIP_DEFLATED) as bundle:
    for name in ("study.json", "settings.json", "resistance_samples.csv", "resistance_summary.csv"):
        path = output / name
        bundle.write(path, name)
    for path in sorted((output / "figures").iterdir()):
        if path.is_file():
            bundle.write(path, "figures/" + path.name)
print("共有するファイル", archive)
from google.colab import files
files.download(str(archive))
